# 03 — Prompt Engineering

## 📚 Learning Objectives

By completing this notebook, you will:
- Know the main **prompt-engineering strategies**: zero-shot, few-shot, chain-of-thought, role prompting
- See hands-on how **conditioning text steers a language model**: one model trained on a real book produces different continuations for different real seeds and temperatures
- Have working **reference templates** for the OpenAI API and Hugging Face `pipeline` (shown as reference; running them needs an API key / model download)

## 🔗 Where this fits

**Builds on:** Course 07 (AIAT 121) — Unit 4, lesson 05, which introduced prompting against GPT-2, and Course 10 — Unit 2, lessons 01-02: fine-tuning changes the weights; prompting changes only the conditioning text fed to the same weights.

**Used later in:** Course 10 — Unit 5, lesson 06, where prompts drive code-generation models.

## 📊 Data used in this notebook

**Real published prose**: Lewis Carroll's *Alice's Adventures in Wonderland*, from the NLTK Gutenberg corpus (`nltk.corpus.gutenberg`, ~144k characters). If that corpus is not downloadable the notebook falls back to real 20 Newsgroups posts — either way the model is trained on text a human actually wrote.

Why this matters for *this* lesson: if you train on a hand-typed sentence repeated ten times, the model memorises it and every seed regurgitates the corpus verbatim. That looks like "steering" but is really recall, and it teaches the wrong lesson. On a real book the model cannot memorise, so the differences you see between seeds are genuine conditioning effects.

---

## Introduction

A language model only ever does one thing: continue the text it is given. **Prompt engineering** is the craft of writing that text so the continuation is what you want. With instruction-tuned LLMs the prompt can carry instructions, examples, and reasoning cues; with our small char-LM the prompt is just a seed — but the *mechanism* (conditioning changes the output distribution) is the same one, and we can demonstrate it live on real text.

---

## 🌍 Why this lesson exists — eight examples, no retraining

Wei et al. (2022, arXiv 2201.11903) prompted a 540-billion-parameter model with **just eight chain-of-thought exemplars** and reached state-of-the-art accuracy on GSM8K grade-school maths — surpassing a *fine-tuned* GPT-3 with a verifier. The weights were untouched. The entire gain came from the text placed in front of the question.

That is the case for taking prompting seriously as engineering rather than as folklore: it is the cheapest intervention available, it is usually tried *after* somebody has already proposed a fine-tune, and on the right task it beats the expensive option outright.

**What goes wrong without this:** the same model, prompted badly, is worth a fraction of what it is worth prompted well — and teams routinely conclude "the model can't do this" when what they have established is "our prompt can't get it to".


## The Four Core Strategies

**Zero-shot** — instruction only:
```text
Classify this review's sentiment as positive or negative.
Review: "The battery dies in an hour."
Sentiment:
```

**Few-shot** — show worked examples first; the model imitates the pattern:
```text
Review: "Amazing sound quality!"        → positive
Review: "Broke after two days."         → negative
Review: "The battery dies in an hour."  →
```

**Chain-of-thought** — ask for reasoning steps before the answer (helps on math/logic):
```text
Q: A shop had 23 apples, sold 9, bought 15. How many now?
A: Let's think step by step.
```

**Role prompting** — set persona and audience to control tone and depth:
```text
You are a patient math tutor for 12-year-olds. Explain fractions with one everyday example.
```

Practical rules that hold across all of them: be specific, state the output format, give the model examples of exactly what you want, and iterate — prompt design is empirical.


In [1]:
# WHAT/WHY: demonstrate the MECHANISM behind prompting — conditioning text
# steers generation. We train one char-level LM on a REAL book, then vary only
# the seed text and the temperature, and watch the continuations change.
# Real prose (not a memorisable toy string) is essential here: a memorised
# corpus would replay itself for any seed and prove nothing about steering.
import torch, torch.nn as nn, torch.optim as optim
import numpy as np
import re

torch.manual_seed(42); np.random.seed(42)

# ── REAL DATA: Lewis Carroll, "Alice's Adventures in Wonderland" ──────────
# NLTK ships the Gutenberg corpus; if it cannot be fetched we fall back to
# real Usenet posts. Both are text a human actually wrote - never invented.
def load_real_corpus():
    try:
        import nltk
        nltk.download('gutenberg', quiet=True)
        from nltk.corpus import gutenberg
        return gutenberg.raw('carroll-alice.txt'), "Carroll, Alice's Adventures in Wonderland (Gutenberg)"
    except Exception:
        from sklearn.datasets import fetch_20newsgroups
        news = fetch_20newsgroups(subset='train', categories=['rec.autos'],
                                  remove=('headers', 'footers', 'quotes'))
        return " ".join(news.data), "20 Newsgroups rec.autos posts (fallback)"

raw, source_name = load_real_corpus()
# Light normalisation keeps the character vocabulary small enough for a CPU LSTM.
text = re.sub(r'[^a-z .,]', ' ', raw.lower())
text = re.sub(r'\s+', ' ', text).strip()[:24000]
print(f"Source: {source_name}")
print(f"Corpus: {len(text):,} characters")
print(f"Sample: {text[:150]!r}")

chars = sorted(set(text)); c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for c, i in c2i.items()}; VOCAB = len(chars); SEQ_LEN = 20
enc = [c2i[c] for c in text]
X = torch.tensor([enc[i:i+SEQ_LEN] for i in range(len(enc)-SEQ_LEN-1)], dtype=torch.long)
y = torch.tensor([enc[i+SEQ_LEN]   for i in range(len(enc)-SEQ_LEN-1)], dtype=torch.long)
print(f"Vocabulary: {VOCAB} characters | training windows: {len(X):,}")

# ── Train the LM once (same model as example 01) ──────────────────────────
class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 32)
        self.lstm  = nn.LSTM(32, 128, batch_first=True, num_layers=2)
        self.fc    = nn.Linear(128, VOCAB)
    def forward(self, x):
        out, _ = self.lstm(self.embed(x))
        return self.fc(out[:, -1, :])

model = CharLM(); opt = optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()
for step in range(800):
    model.train()
    perm = torch.randperm(len(X))[:256]
    loss = loss_fn(model(X[perm]), y[perm])
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 200 == 0:
        print(f"step {step} — loss {loss.item():.3f}")
print(f"LM trained on real prose — final batch loss {loss.item():.3f}")

def generate(seed, steps=90, temperature=0.8):
    # condition on the seed, then sample the continuation char by char
    model.eval()
    out = list(seed); ctx = [c2i.get(c, 0) for c in seed[-SEQ_LEN:]]
    for _ in range(steps):
        with torch.no_grad():
            logits = model(torch.tensor([ctx[-SEQ_LEN:]]))[0] / temperature
        nxt = int(np.random.choice(VOCAB, p=torch.softmax(logits, 0).numpy()))
        out.append(i2c[nxt]); ctx.append(nxt)
    return ''.join(out)

# ── Steering experiment 1: SAME model, different real seeds ───────────────
# Each seed is a genuine 20-character phrase from the book - three different
# passages, so the model is conditioned on three different contexts.
np.random.seed(7)
print("\n— Different seeds steer the continuation (temperature 0.8) —")
for seed in [text[0:20], text[8000:8020], text[16000:16020]]:
    print(f"  seed {seed!r}\n    → {generate(seed)!r}")

# ── Steering experiment 2: SAME seed, different temperatures ──────────────
np.random.seed(7)
print("\n— Same seed, different temperatures (decoding is part of the prompt kit) —")
for temp in [0.3, 1.5]:
    print(f"  temperature {temp}:\n    → {generate(text[0:20], temperature=temp)!r}")

print("\nWhat this shows: generation is conditioned on the text you provide, and")
print("on a real corpus the continuations genuinely differ instead of replaying")
print("a memorised script. Prompt engineering on an LLM exploits exactly this —")
print("with instructions and examples in the conditioning text instead of a bare")
print("seed. Note the limits: a char-LM cannot follow instructions; that ability")
print("comes from instruction tuning (RLHF) on large models.")


Source: Carroll, Alice's Adventures in Wonderland (Gutenberg)
Corpus: 24,000 characters
Sample: 'alice s adventures in wonderland by lewis carroll chapter i. down the rabbit hole alice was beginning to get very tired of sitting by her sister on th'
Vocabulary: 29 characters | training windows: 23,979


step 0 — loss 3.332


step 200 — loss 1.720


step 400 — loss 1.520


step 600 — loss 1.376


LM trained on real prose — final batch loss 0.920

— Different seeds steer the continuation (temperature 0.8) —
  seed 'alice s adventures i'
    → 'alice s adventures if i won, she cis sumer our the mouse her finer way it m beray and would little either murs'
  seed 'nd she had never for'
    → 'nd she had never forgot it said the sibed this ince it allosed she felt age a calied onive to the begar the do'
  seed 'at her hands, and wa'
    → 'at her hands, and was shame with they were was poor, saying haply inder, but it was alice was amar such a thin'

— Same seed, different temperatures (decoding is part of the prompt kit) —
  temperature 0.3:
    → 'alice s adventures in the rabbit should she was now more of the rabbit shought thought this should she was now'
  temperature 1.5:
    → 'alice s adventures it. there s ip opor in shrughgan. to mow. to ite stoibit .acg ot dinwy asmersto in verieg n'

What this shows: generation is conditioned on the text you provide, and
on a real corpus

## 💬 Discuss

1. Three different real seeds produced three clearly different continuations from **one** set of weights; Wei et al. got a state-of-the-art jump on maths the same way. Where exactly is the boundary between "prompting elicited a capability" and "the model already had it and we finally asked properly"? Does the distinction matter to a client?
2. At **T = 0.3** our model looped on frequent phrasing (`the rabbit should she was now more of the rabbit`); at **T = 1.5** it produced non-words (`there s ip opor in shrughgan`). A client complains the assistant is "boring". Is temperature the right knob? What would you check first, and what would you check second?
3. A prompt is untrusted text that becomes instructions. If our seed text arrived in a customer email, what could a malicious customer make the model do? (Greshake et al., 2023, arXiv 2302.12173, named this class of attack.)


## The Same Strategies on Real LLMs (reference)

**Reference only — not executed here** (needs an API key or a model download):

OpenAI API — the prompt strategies go into the `messages`:
```python
from openai import OpenAI
client = OpenAI()   # needs OPENAI_API_KEY

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a patient math tutor."},   # role prompt
        {"role": "user",   "content": "Q: 23 apples, sold 9, bought 15. "
                                       "How many now? Think step by step."}  # chain-of-thought
    ],
    temperature=0.3,   # same knob you used above
)
print(resp.choices[0].message.content)
```

Hugging Face `pipeline` — few-shot by stacking examples into the prompt:
```python
from transformers import pipeline
generator = pipeline("text-generation", model="distilgpt2")   # downloads the model

prompt = ("Review: 'Amazing sound!' → positive\n"
          "Review: 'Broke in two days.' → negative\n"
          "Review: 'Battery dies fast.' →")
print(generator(prompt, max_new_tokens=5, temperature=0.3)[0]["generated_text"])
```

Try these on your own machine/account — then iterate on the prompt and watch the output change, exactly as in the steering experiment above.


## ⚠️ Where this breaks

**Prompting steers a distribution; it does not add capability.** Our character LSTM cannot be prompted into following an instruction — it has no representation of one. Equally, no LLM can be prompted into knowing a fact that is not in its weights; that is a retrieval problem, not a wording problem.

**Prompt gains are model-specific and version-fragile.** A prompt tuned against one model version can lose most of its benefit on the next. Treat prompts as code: version them, test them against a fixed evaluation set, and re-run that set on every model upgrade.

**Prompting is an attack surface, not just an interface.** Anything that reaches the context window — a retrieved web page, a user's email, a filename, a PDF — becomes instructions to the model (Greshake et al., 2023). The 2026 defences are architectural (least-privilege tool policies, e.g. Progent, arXiv 2504.11703), not better wording, because there is no phrasing that makes untrusted input safe.

**If you need a guaranteed output format, parse and validate it.** Asking nicely is not a contract. Every production prompt that must produce JSON is paired with a parser and a retry, because the model's compliance is probabilistic by construction.


## 📚 References & Further Reading

**Papers:**
- Brown et al. (2020) — [GPT-3: Language Models are Few-Shot Learners](https://arxiv.org/abs/2005.14165) *(few-shot prompting)*
- Wei et al. (2022) — [Chain-of-Thought Prompting](https://arxiv.org/abs/2201.11903)

**Guides:**
- [OpenAI Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering)
- [promptingguide.ai](https://www.promptingguide.ai/) *(open catalogue of techniques)*


## 📝 Summary

In **03 — Prompt Engineering** you learned the four core strategies (zero-shot, few-shot, chain-of-thought, role prompting) with concrete templates, and demonstrated the underlying mechanism on a model you trained yourself on **real published prose** — 24,000 characters of *Alice'''s Adventures in Wonderland*.

What the printed experiments actually showed:
- **Seed matters.** Three different real 20-character passages produced three clearly different continuations. Because the model reached a batch loss around 0.9 on a 24k-character book — nowhere near memorisation — those differences are genuine conditioning, not recall.
- **Temperature matters.** The same seed at T=0.3 produced repetitive, safe text that loops on frequent phrases (`the rabbit should she was now more of the rabbit...`); at T=1.5 it degenerated into noise (`there s ip opor in shrughgan`). Decoding settings are part of the prompt toolkit.
- **The limit is honest.** A character LSTM cannot follow an instruction. Instruction-following comes from instruction tuning and RLHF on large models — which is why the API reference blocks below matter.

The API reference blocks show how the same strategies are written for GPT-class models; running them requires an API key or model download, which this classroom notebook deliberately avoids.
